# Giai đoạn 3 — Huấn luyện trên GPU Google Colab
Không chạy train trên máy cá nhân. Runtime → Change runtime type → T4 GPU. MLP và CNN: 54k train / 6k validation, mỗi model 5 epoch; chưa xem test. Mã được tải từ GitHub, không chép vòng train vào notebook.

In [ ]:
import sys, subprocess, json, shutil
from pathlib import Path
from uuid import uuid4
import google.colab
import torch
assert torch.cuda.is_available(), 'Cần bật GPU; không chuyển sang CPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
PROJECT = Path('/content') / ('fashion_stage3_' + uuid4().hex[:8])
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/tung2106/Fashion_MNIST_CNN-MLP.git', str(PROJECT)], check=True)
STAGE = PROJECT / '03_Thuc_Nghiem_Cai_Tien'
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)


## 2. Kiểm tra thư viện
Dùng bộ PyTorch/CUDA có sẵn của Colab, không cài lại torch. Các thư viện còn lại được import để phát hiện thiếu trước khi train.

In [ ]:
import torchvision, numpy, pandas, sklearn, matplotlib
print('Thư viện sẵn sàng. CUDA:', torch.cuda.is_available())


## 3. Train đầy đủ MLP rồi CNN
Adam 0.001, batch 128, 5 epoch/model. Chọn best theo validation loss thấp nhất. Không có cờ --evaluate-test nên chưa dùng test. Lệnh chạy trên máy ảo GPU của Google, không trên máy bạn.

In [ ]:
before = set((STAGE / 'artifacts').glob('*')) if (STAGE / 'artifacts').exists() else set()
subprocess.run([sys.executable, '-u', str(STAGE / 'main.py')], check=True)
created = set((STAGE / 'artifacts').glob('*')) - before
assert len(created) == 1
RUN = created.pop()
status = json.loads((RUN / 'STATUS.json').read_text())
assert status['stage'] == 'completed' and not status['test_evaluated']
(RUN / 'source_commit.txt').write_text(COMMIT)
print('TRAIN_COMPLETE:', RUN)


## 4. Đọc kết quả validation
Đây không phải accuracy test. Best checkpoint có thể ở epoch trước epoch cuối.

In [ ]:
from IPython.display import display, Image
for name in ['MLP', 'CNN']:
    print(name, json.loads((RUN / 'metrics' / f'{name}_training.json').read_text()))
    display(pandas.read_csv(RUN / 'metrics' / f'{name}_history.csv'))
    display(Image(filename=str(RUN / 'figures' / f'{name}_learning_curve.png')))


## 5. Lưu kết quả về máy
ZIP chứa checkpoint, cấu hình, split và biểu đồ. Notebook trên Drive không tự giữ file trong máy ảo; tải ZIP để giữ chúng.

In [ ]:
from google.colab import files
archive = shutil.make_archive(str(RUN), 'zip', root_dir=RUN)
print('Kết quả:', archive)
files.download(archive)
